# Task 1 — Reproducing the official baseline

Goal (Act 1): recompute the three indicators of `baseline_officielle.csv` from the raw data, following the
definitions of `docs/notice_metier.md`, and understand any gap before moving on.

Rules applied (from the guide, not free choices):
- **Response time** = alert → arrival of the **first vehicle to arrive**, whatever its type.
- **Unfulfilled** intervention (empty arrival time) → conventional delay of **60 min**, kept in the delay indicators.
- **Compliance** = delay ≤ threshold(zone, reason); **Z4 has no binding target** and is excluded from the rate.

Raw files live in `raw_data/` (byte-identical copy of `donnees/`, never modified). Shared code is in `helpers.py`.

In [1]:
import pandas as pd
import numpy as np
import helpers as h

pd.set_option("display.width", 160)
raw = h.load_raw()
pd.DataFrame({name: df.shape for name, df in raw.items()}, index=["rows", "columns"]).T

,rows,columns
interventions,74215,9
communes,120,5
centres,30,6
plan_deploiement,960,3
seuils_reglementaires,12,3
baseline_officielle,3,2


## 1. Brief inspection — only what the calculation needs

In [2]:
iv = raw["interventions"]
print("vehicle rows:", len(iv), "| interventions:", iv["id_intervention"].nunique())
print("vehicles per intervention:", iv.groupby("id_intervention").size().value_counts().sort_index().to_dict())
print("exact duplicate rows:", iv.duplicated().sum())

# One intervention = one alert, one commune, one reason?
g = iv.groupby("id_intervention")
print("ids with >1 alert time / commune / reason:",
      (g["horodatage_alerte"].nunique() > 1).sum(), (g["insee"].nunique() > 1).sum(), (g["raison"].nunique() > 1).sum())
pd.crosstab(iv["raison"], iv["type_engin"], margins=True)

vehicle rows: 74215 | interventions: 55000
vehicles per intervention: {1: 42204, 2: 6377, 3: 6419}
exact duplicate rows: 0
ids with >1 alert time / commune / reason: 0 0 0


type_engin,CCR,FPT,VSAV,VTU,All
raison,,,,,
AVP,855,841,7390,861,9947
INC,1933,11981,1978,2028,17920
SAP,1880,1966,40501,2001,46348
All,4668,14788,49869,4890,74215


**Missing arrivals.** The guide says an unfulfilled intervention appears with an empty arrival time.
Check that these rows are whole interventions (no other vehicle arrived) and see their reasons.

In [3]:
missing = iv[iv["horodatage_arrivee"].isna()]
ids_missing = missing["id_intervention"].unique()
other_rows = iv[iv["id_intervention"].isin(ids_missing) & iv["horodatage_arrivee"].notna()]
print("rows with empty arrival:", len(missing), "| interventions concerned:", len(ids_missing),
      "| of which a vehicle still arrived:", other_rows["id_intervention"].nunique())
missing.groupby(["raison", "type_engin"]).size().rename("unfulfilled interventions")

rows with empty arrival: 672 | interventions concerned: 672 | of which a vehicle still arrived: 0


raison  type_engin
AVP     VSAV           60
INC     FPT           290
SAP     VSAV          322
Name: unfulfilled interventions, dtype: int64

**No dispatch-rank column.** The guide mentions a rank, but the file has none, and the file order is not
the arrival order (example below: the VTU is listed first, the FPT arrives first). So the first vehicle must be found
by **earliest arrival time**, not by row order.

In [4]:
iv[iv["id_intervention"] == "I000003"]

,id_intervention,horodatage_alerte,insee,raison,type_engin,code_cis,delai_depart_min,duree_trajet_s,horodatage_arrivee
3,I000003,2025-01-01 01:16:58,25001,INC,VTU,CIS23,3.54,1290.0,2025-01-01 01:42:00
4,I000003,2025-01-01 01:16:58,25001,INC,FPT,CIS07,1.67,526.0,2025-01-01 01:27:24


In [5]:
# Is the earliest-arriving vehicle ever something other than VSAV / FPT? (guide: other types never respond first)
arrived = iv.dropna(subset=["horodatage_arrivee"])
first_rows = arrived.sort_values("horodatage_arrivee").drop_duplicates("id_intervention")
first_rows["type_engin"].value_counts()

type_engin
VSAV    44644
FPT      9684
Name: count, dtype: int64

In [6]:
# Timestamps vs duration columns: arrival - alert should equal delai_depart_min + duree_trajet_s / 60
ts = (arrived["horodatage_arrivee"] - arrived["horodatage_alerte"]).dt.total_seconds()
dur = arrived["delai_depart_min"] * 60 + arrived["duree_trajet_s"]
print(f"max |timestamps - durations| = {(ts - dur).abs().max():.1f} s  (timestamps are rounded to the second)")
raw["seuils_reglementaires"].pivot(index="zone_sdacr", columns="raison", values="seuil_min")

max |timestamps - durations| = 1.6 s  (timestamps are rounded to the second)


raison,AVP,INC,SAP
zone_sdacr,,,
Z1,40.0,20.0,20.0
Z2,40.0,25.0,20.0
Z3,40.0,30.0,20.0
Z4,NaN,NaN,NaN


Inspection summary:
- 74 215 vehicle rows for 55 000 interventions; no exact duplicate rows; each id has a single alert, commune and reason.
- 672 interventions have an empty arrival, each is a single row with no other vehicle → these are the unfulfilled ones.
- No rank column and row order ≠ arrival order → first vehicle = earliest arrival.
- Only VSAV and FPT ever arrive first, as the guide states, so keeping all vehicle types changes nothing.
- Timestamps and duration columns agree to within 1.6 s (rounding). Z4 thresholds are empty (no binding target).

## 2. One row per intervention, judged on its first-arriving vehicle

In [7]:
interventions = h.build_interventions(raw)          # delay = arrival timestamp - alert timestamp
h.DATA.mkdir(exist_ok=True)
interventions.to_csv(h.DATA / "first_arrivals.csv", index=False)
print(interventions.shape)
interventions.head()

(55000, 13)


,id_intervention,alert_time,insee,reason,n_vehicles,first_vehicle,first_centre,observed_delay_min,unfulfilled,delay_min,zone,threshold_min,compliant
0,I000000,2025-01-01 00:28:20,25106,SAP,1,VSAV,CIS07,6.250000,False,6.250000,Z2,20.0,True
1,I000001,2025-01-01 00:30:36,25082,SAP,1,VSAV,CIS27,3.983333,False,3.983333,Z1,20.0,True
2,I000002,2025-01-01 00:40:08,25021,SAP,1,VSAV,CIS23,13.366667,False,13.366667,Z2,20.0,True
3,I000003,2025-01-01 01:16:58,25001,INC,2,FPT,CIS07,10.433333,False,10.433333,Z2,25.0,True
4,I000004,2025-01-01 01:29:08,25007,SAP,1,VSAV,CIS22,5.366667,False,5.366667,Z2,20.0,True


## 3. The three indicators vs the official baseline

In [8]:
kpis = h.compute_kpis(interventions)
check = h.compare_to_baseline(kpis, raw["baseline_officielle"])
h.RESULTS.mkdir(exist_ok=True)
check.round(4).to_csv(h.RESULTS / "kpi_check.csv")
check.round(4)

,official,recomputed,gap
indicator,,,
p90_delai_sap_min,25.51,25.5100,-0.0000
inc_non_honorees,290.00,290.0000,0.0000
taux_conformite_pct,86.45,86.4605,0.0105


## 4. Where does the 0.01-point compliance gap come from?

`p90_delai_sap_min` and `inc_non_honorees` match exactly. The compliance rate is 86.46 % against 86.45 %
(≈ 5 interventions out of 53 525). Same calculation, but with the delay taken from the duration columns
(`delai_depart_min + duree_trajet_s / 60`, precise to a fraction of a second) instead of the timestamps:

In [9]:
interventions_dur = h.build_interventions(raw, delay_source="durations")
check_dur = h.compare_to_baseline(h.compute_kpis(interventions_dur), raw["baseline_officielle"])
pd.concat({"timestamps": check["recomputed"], "durations": check_dur["recomputed"],
           "official": check["official"]}, axis=1).round(4)

,timestamps,durations,official
indicator,,,
p90_delai_sap_min,25.5100,25.5033,25.51
inc_non_honorees,290.0000,290.0000,290.00
taux_conformite_pct,86.4605,86.4512,86.45


In [10]:
# Interventions whose compliance verdict depends on the delay source
both = interventions.set_index("id_intervention")[["reason", "zone", "threshold_min", "delay_min"]].join(
    interventions_dur.set_index("id_intervention")["delay_min"].rename("delay_min_durations"))
flips = both[both["threshold_min"].notna()
             & ((both["delay_min"] <= both["threshold_min"]) != (both["delay_min_durations"] <= both["threshold_min"]))]
flips.assign(delay_s=flips["delay_min"] * 60, delay_s_durations=flips["delay_min_durations"] * 60).round(2)

,reason,zone,threshold_min,delay_min,delay_min_durations,delay_s,delay_s_durations
id_intervention,,,,,,,
I021374,INC,Z2,25.0,25.00,25.00,1500.0,1500.2
I022955,SAP,Z2,20.0,20.00,20.01,1200.0,1200.6
I030359,SAP,Z2,20.0,20.00,20.01,1200.0,1200.4
I040745,SAP,Z3,20.0,20.00,20.01,1200.0,1200.6
I045773,INC,Z3,30.0,30.02,30.00,1801.0,1800.0
I047452,SAP,Z2,20.0,20.00,20.01,1200.0,1200.4
I052951,INC,Z2,25.0,25.00,25.01,1500.0,1500.8


Reading of the gap:
- 7 interventions are **within one second of their threshold**. Six are exactly on it when rounded to the second
  (so compliant), but 0.2–0.8 s over it in the duration columns; one is 1 s over in the timestamps but exactly on it
  in the durations. Net: 5 fewer compliant interventions with the durations.
- With the duration columns the compliance rate is **86.45 % (exact)**, but the SAP P90 becomes 25.50 instead of 25.51.
- So neither precision reproduces all three to the last digit; each residual is a **sub-second rounding effect**
  (≤ 0.01 point, ≤ 0.4 s), which the brief calls a negligible gap. The calculation itself (first arrival, 60-min
  convention, Z4 exclusion, zone × reason thresholds) is confirmed.
- Kept as reference: **timestamps**, the guide's literal definition (alert → arrival), which reproduces P90 and the
  unfulfilled-fire count exactly.